#### 로컬 임베딩 모델 사용하기

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from glob import glob

for g in glob('./data/*.pdf'):
    print(g)


./data\2040_seoul_plan.pdf
./data\OneNYC_2050_Strategic_Plan.pdf


In [3]:
# 임베딩 모델 설치
# pip install langchain-huggingface

# 파이토치 설치하기(option)
# CUDA를 지원하는 NVIDIA 그래픽 카드를 갖고 있다면 파이토치를 설치
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [4]:
# 청크 분할하기
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    매개변수:
        pdf_path (str): 파일의 경로
        chunk_size (int, 선택적): 각 텍스트 청크의 크기. 기본값은 1000입니다.
        chunk_overlap(int, 선택적): 청크 간의 중첩 크기. 기본값은 100입니다.
    반환값:
        list: 분할된 텍스트 청크의 리스트    
    """
    print(f'PDF: {pdf_path} ------------------')

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)
    print(f'Number of splits: {len(splits)}')

    return splits

C:\Users\admin\AppData\Local\Temp\ipykernel_7936\1606121706.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\workspaces\ai_agent\src\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs = {'device': 'cpu'},               # cuda 지원하면 cuda, 아닌 경우 cpu
    encode_kwargs = {'normalize_embeddings': True}, # 임베딩 정규화
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 33693.69it/s]


In [6]:
# 임베딩 모델 테스트
embeddings.embed_documents('안녕하세요.')

[[0.002456831047311425,
  0.0322626568377018,
  -0.007424292620271444,
  0.00526847131550312,
  -0.05809730291366577,
  -0.030428674072027206,
  -0.008300005458295345,
  0.036742474883794785,
  0.009622779674828053,
  -0.007633916102349758,
  0.017204467207193375,
  0.038877058774232864,
  -0.02964903600513935,
  -0.021820029243826866,
  -0.0008699100580997765,
  -0.032562505453825,
  0.03178011626005173,
  -0.022462930530309677,
  0.023502597585320473,
  -0.012333741411566734,
  -0.038810186088085175,
  -0.017907707020640373,
  0.038729410618543625,
  0.013715686276555061,
  0.025562595576047897,
  0.0229959636926651,
  -0.02759172022342682,
  0.027073416858911514,
  -0.009775801561772823,
  -0.026509825140237808,
  -0.006837860215455294,
  -0.02930246852338314,
  0.028823185712099075,
  -0.07535599172115326,
  -0.03274378553032875,
  -0.004341255873441696,
  -0.023116106167435646,
  0.02154514007270336,
  -0.054298121482133865,
  0.05281256139278412,
  0.04333128780126572,
  -0.02088

In [7]:
# pdf 파일 읽고 텍스트 임베딩하기
from langchain_chroma import Chroma
import os

persist_directory = './chroma_store'  # 벡터 DB

if os.path.exists(persist_directory):
    print('Loading existing Chroma store')

    vectorstore = Chroma(  # 읽어 올때
        persist_directory=persist_directory,
        embedding_function=embeddings
    )

else:
    print('Creating new Chroma store')

all_splits= []
for g in glob('./data/*.pdf'):
    all_splits.extend(read_pdf_and_split_text(g))

    print(f'Total number of splits: {len(all_splits)}')

    vectorstore = Chroma.from_documents(  # 저장(생성)할 때 : 최초 한번만 호출되어야 함
        documents=all_splits,
        embedding=embeddings,  # BAAI/bge-m3 모델
        persist_directory=persist_directory
    )

Creating new Chroma store
PDF: ./data\2040_seoul_plan.pdf ------------------
Number of splits: 308
Total number of splits: 308
PDF: ./data\OneNYC_2050_Strategic_Plan.pdf ------------------
Number of splits: 1023
Total number of splits: 1331


In [8]:
# 리트리버 테스트(검색기)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5}) # 5개의 chunk 데이터를 반환

chunks = retriever.invoke('서울시 쓰레기 저감 정책')

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)


{'source': './data\\2040_seoul_plan.pdf', 'pdfversion': '1.4', 'creationdate': '2024-12-12T18:16:11+09:00', 'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2020 11.0.0.5178', 'total_pages': 205, 'moddate': '2024-12-12T18:16:11+09:00', 'author': 'SI', 'page_label': '65', 'page': 64}
제3절 2040 서울도시기본계획 7대 목표57Ÿ특히, 서울시는 현재 온실가스 배출량의 90%를 차지하고 있는 건물과 수송 부문 감축을 위해 적극적인 대책을 마련하고 있다.-2026년까지 건물 에너지효율화사업 100만 호를 추진, 건물온실가스총량제, 신규건물 제로에너지 건물(ZEB) 의무화를 통해 기존 및 신규건물의 제로에너지화를 촉진-수송 부문 배출감축을 위해 전기차 비중을 2026년까지 10%(’21년 5.2만 대 → ’26년 40만 대)로 확대하고, 22만기의 충전 인프라 구축을 계획 중(’21년 2만기) 시민 생활과 안전을 위한 기후위기 대응 요구 심화Ÿ서울은 인구, 시설 등이 밀집해 있는 대도시로 기온 상승, 폭염, 집중호우, 태풍, 한파 등 극한 기후 현상이 더욱 빈번하게 발생할 것으로 전망된다. 이러한 기후위험은 서울시민의 일상생활과 안전을 크게 위협할 수 있다. 따라서 시민의 일상을 보호하고 도시의 회복력을 강화하기 위한 적극적인 기후위기 대응 전략이 필요하다.Ÿ특히 다양화·복합화되는 재난안전사고에 대응해 전통적인 자연·사회재난의 범주뿐 아니라, 신종 복합재난까지 대비할 수 있는 다면적인 대응체계 마련이 요구된다.2) 추진전략탄소중립·기후위기 적응대책은 시의 모든 정책과 사업의 주요 원칙으로 고려Ÿ탄소중립은 돌이키기 어려운 기후재난을 막기 위한 국제사회와의 약속이기에 시의 모든 정책과 사업의 결정 과정에서 주요하게 고려해야 할 포괄적 ‘원칙’으

In [16]:
# tool 데코레이터등록(RAG를 도구로 등록)
from langchain_core.tools import tool

@tool
def search_docs(query: str) -> str:
    """Vector store에서 질문(query)과 관련된 문서 Chunk를 검색하여 텍스트로 반환합니다.

    Args:
        query (str): 검색할 질문 또는 키워드
    """
    retriever = vectorstore.as_retriever(
        search_kwargs={'k': 5}
    ) # 5개의 chunk 데이터를 반환

    results = retriever.invoke(query)
    # chunks = retriever.invoke('서울시 쓰레기 저감 정책')

    # LLM이 읽기 쉬운 형태로 컨텍스트 문자열 구성
    # results = []

    # for i, chunk in enumerate(chunks, 1):
    #     content = chunk.page_content
    #     metedata = chunk.metadata
    #     # 메타데이터와 본문을 결합하여 개별 chunk 포맷팅
    #     chunk_text = f'[문서 {i}\n메타데이터 {metedata} 내용: {content}]'
    #     results.append(chunk_text)

    # 5개의 검색 결과를 구분선으로 연결하여 하나의 문자열로 반환
    # return '\n\n-----------\n\n'.join(results)
    return results


In [17]:
# RAG(검색 증강 생성) 에이전트
# RAG을 tool로 등록
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-5-nano', temperature=0)

tools = [search_docs]

system_prompt = """
    당신은 업로드된 PDF 문서의 내용을 기반으로 사용자의 질문에 정확하게 답변하는 전문 문서 분석 AI 에이전트입니다.

    다음 지침을 엄격히 준수하여 답변하세요:
    1. search_docs 도구를 사용하여 관련 정보를 찾으세요.
    2. 검색된 문서 내용(Context)만을 바탕으로 답변해야 하며, 당신이 알고 있는 외부 지식이나 추측을 바탕으로 답변을 작성하지 마세요.
    3. 검색 결과에 질문에 대한 답이 없거나 정보가 부족한 경우, 지어내지 말고 "제시된 문서 내용에서 해당 질문에 대한 정보를 찾을 수 없습니다."라고 명확히 답변하세요.
    4. 답변 시 관련 정보가 추출된 문서의 페이지 번호나 출처 메타데이터가 있다면 함께 언급해 주세요.
    5. 간결하고, 명확하며, 격식 있는 어조로 한국어로 작성하세요.
"""

# RAG Agent 생성
rag_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt
)

query = '서울시 쓰레기 저감 정책은 어떻게 되나요?'
messages = [{
    'role': 'user',
    'content': query
}]

print(f'사용자 질문 {query}')


for chunk in rag_agent.stream({
    "messages": messages
}, stream_mode="values"):
    
    latest_msg = chunk['messages'][-1]

    if latest_msg.__class__.__name__ == "AIMessage":
        if latest_msg.content:
            print(f'Agent 생각:\n{latest_msg.content[:150]}...\n')

        if latest_msg.tool_calls:
            for tc in latest_msg.tool_calls:
                print(f'도구 호출: {tc['name']}')    
                print(f'     입력: {tc['args']}\n')    

    elif latest_msg.__class__.__name__ == "ToolMessage":
        print(f'도구 결과')
        print(f'   {latest_msg.content[:200]}...\n')

print('\n최종 응답')
print(chunk['messages'][-1].content)



사용자 질문 서울시 쓰레기 저감 정책은 어떻게 되나요?
도구 호출: search_docs
     입력: {'query': '서울시 쓰레기 저감 정책'}

도구 결과
   [Document(id='e7f92907-5970-426b-aaa9-2f0fc563ad80', metadata={'creationdate': '2024-12-12T18:16:11+09:00', 'producer': 'Hancom PDF 1.3.0.542', 'pdfversion': '1.4', 'page': 64, 'creator': 'Hwp 2020 11...

Agent 생각:
다음 자료에 근거하여 요약합니다.

요약
- 서울시 쓰레기 저감은 2040 서울도시기본계획의 자원순환 부문으로 다루어지며, 탄소중립 및 기후위기 대응 전략의 핵심 원칙으로 모든 정책·사업에 반영하는 방향으로 제시됩니다.
- 구체적 지표로는 자원순환 관련 목표를 제시하고...


최종 응답
다음 자료에 근거하여 요약합니다.

요약
- 서울시 쓰레기 저감은 2040 서울도시기본계획의 자원순환 부문으로 다루어지며, 탄소중립 및 기후위기 대응 전략의 핵심 원칙으로 모든 정책·사업에 반영하는 방향으로 제시됩니다.
- 구체적 지표로는 자원순환 관련 목표를 제시하고 있으며, 인구당 생활폐기물 발생량(매립률) 등 쓰레기 관리의 수치를 설정하고 있습니다. 특히 2027년까지 인구 1인당 생활폐기물 발생량을 0.86 kg/인/일로 관리하는 것을 목표로 제시합니다. 이는 매립률 관련 지표로도 연결됩니다.

주요 내용과 출처
- 2040 서울도시기본계획의 제3절 7대 목표 부문에서 기후위기 대응과 자원순환 정책의 원칙을 시의 모든 정책에 반영한다는 추진 원칙이 제시되어 있습니다. 페이지 표기상 문서의 해당 섹션은 페이지 65쪽에 해당합니다. 출처: 2040 서울도시기본계획(제3절 7대 목표), 페이지 65.
- 동시에 제6장 계획의 실현 부분의 표에서 자원순환 관련 지표로 “인구당 생활폐기물 발생량(매립률)”을 제시하고 있으며, 구체적